In [ ]:
import pandas as pd
import re
df = pd.read_csv('../data/raw/tech_jobs_2026.csv')

In [11]:
df.shape

(23201, 32)

In [12]:

SENIORITY_TOKENS = {
    "senior", "sr", "sr.", "junior", "jr", "jr.", "lead", "principal",
    "graduate", "experienced", "expert", "i", "ii", "iii", "iv",
}

def normalize_title(raw_title) -> str:
    t = str(raw_title).strip()
    t = re.sub(r"\([^)]*\)", "", t)
    t = re.sub(r"\[[^\]]*\]", "", t)
    t = re.sub(r"[\-\u2013\u2014/|:]+", " ", t)

    tokens = t.split()
    while tokens and tokens[0].strip(",.").lower() in SENIORITY_TOKENS:
        tokens.pop(0)
    while tokens and tokens[-1].strip(",.").lower() in SENIORITY_TOKENS:
        tokens.pop()

    cleaned = " ".join(tokens).strip()
    return cleaned if cleaned else str(raw_title).strip()


def canonical_key(normalized_title: str) -> str:
    k = normalized_title.lower()
    k = re.sub(r"[^a-z0-9+/]+", " ", k)
    k = re.sub(r"\s+", " ", k).strip()

    replacements = [
        (r"\bfull\s*stack\b", "fullstack"),
        (r"\bdev\s*ops\b", "devops"),
        (r"\bml\s*ops\b", "mlops"),
        (r"\bback\s*end\b", "backend"),
        (r"\bfront\s*end\b", "frontend"),
        (r"\bgen\s*ai\b", "genai"),
        (r"\bai\s*/\s*ml\b", "ai/ml"),
        (r"\bai\s+ml\b", "ai/ml"),
        (r"\bartificial intelligence\b", "ai"),
        (r"\bmachine learning\b", "ml"),
        (r"\bdevelopers\b", "developer"),
        (r"\bengineers\b", "engineer"),
        (r"\bscientists\b", "scientist"),
        (r"\banalysts\b", "analyst"),
        (r"\bconsultants\b", "consultant"),
        (r"\bsoftware developer\b", "software engineer"),
        (r"\bsoftware tester\b", "qa engineer"),
        (r"\btest engineer\b", "qa engineer"),
        (r"\bquality engineer\b", "qa engineer"),
        (r"\bquality analyst\b", "qa engineer"),
        (r"\bquality assurance engineer\b", "qa engineer"),
        (r"\bsdet\b", "qa engineer"),
    ]

    for p, r in replacements:
        k = re.sub(p, r, k)

    return k

df["canonical_key"] = df["job_title"].apply(normalize_title).apply(canonical_key)

TARGET_CLASSES = [
    "AI/ML Engineer", "Data Scientist", "Data Analyst", "Python Developer",
    "Software Engineer", "Full Stack Developer", "QA Engineer", "DevOps Engineer",
]

def map_to_target(key: str):
    if re.search(r"\b(ai/ml|aiml|ai|ml|machine learning)\b", key) and re.search(r"\bengineer\b", key):
        return "AI/ML Engineer"

    if re.search(r"\b(data scientist)\b", key):
        return "Data Scientist"

    if re.search(r"\b(data analyst)\b", key):
        return "Data Analyst"

    if re.search(r"\bpython\b", key) and re.search(r"\b(developer)\b", key):
        return "Python Developer"

    if re.search(r"\b(fullstack|mern|mean)\b", key) and re.search(r"\b(developer|engineer)\b", key):
        return "Full Stack Developer"

    if re.search(r"\b(qa engineer|qa|quality assurance|quality engineer|quality analyst|software tester)\b", key):
        return "QA Engineer"

    if re.search(r"\b(devops|infrastructure engineer)\b", key):
        return "DevOps Engineer"

    if re.search(r"\bsoftware engineer\b", key):
        if not re.search(r"\b(ai|ml|data|python|qa|test|devops|fullstack|genai|llm|rag)\b", key):
            return "Software Engineer"

    return None

df["mapped_job_title"] = df["canonical_key"].apply(map_to_target)

matched = df[df["mapped_job_title"].notna() & (df["skills_count"] > 0)].copy()

matched["experience"] = (
    (matched["experience_min_yrs"].fillna(0) + matched["experience_max_yrs"].fillna(0)) / 2
)

out = matched[["mapped_job_title", "skills_required", "skills_count", "experience"]].copy()
out.columns = ["job_title", "skills_required", "skills_count", "experience"]
out = out.reset_index(drop=True)

counts = out["job_title"].value_counts().reindex(TARGET_CLASSES, fill_value=0)
pct = (counts / counts.sum() * 100).round(1)
summary = pd.DataFrame({"job_title": counts.index, "count": counts.values, "pct": pct.values})
out.shape

(6154, 4)

In [ ]:
out.to_csv("../data/raw/tech_jobs_dataset_cleaned.csv", index=False)


Saved tech_jobs_real_8class.csv (6154 rows, 4 columns matching target format)
